# 🎵 Teachable Machine Pose – Song Recognition

This notebook uses the original Teachable Machine Pose browser runtime. It does **not** use `tensorflowjs` or Keras conversion.


In [ ]:
from pathlib import Path
import shutil, json

BASE = Path.cwd()
MODEL_DIR = BASE / 'my_model'
MODEL_DIR.mkdir(exist_ok=True)

for name in ['model.json', 'metadata.json', 'weights.bin']:
    src = BASE / name
    dst = MODEL_DIR / name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

missing = [x for x in ['model.json','metadata.json','weights.bin'] if not (MODEL_DIR/x).exists()]
if missing:
    print('❌ Missing:', missing)
    print('Put model.json, metadata.json and weights.bin beside this notebook.')
else:
    print('✅ Model files found:', MODEL_DIR)
    with open(MODEL_DIR/'metadata.json', encoding='utf-8') as f:
        m=json.load(f)
    print('Classes:')
    for i,x in enumerate(m['labels']): print(i, '->', str(x).strip())


In [ ]:
html = '<!DOCTYPE html>\n<html>\n<head>\n<meta charset="UTF-8">\n<title>Dance Pose Song Recognition</title>\n<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@1.7.4/dist/tf.min.js"></script>\n<script src="https://cdn.jsdelivr.net/npm/@teachablemachine/pose@0.8/dist/teachablemachine-pose.min.js"></script>\n<style>\nbody{font-family:Arial;background:#18263a;color:white;text-align:center;padding:20px}\nbutton{padding:12px 24px;margin:8px;border:0;border-radius:8px;font-size:16px}\n#start{background:#28a745;color:white}#stop{background:#dc3545;color:white}\ncanvas{width:500px;max-width:95%;border:3px solid white;border-radius:12px}\n#song{font-size:28px;font-weight:bold;margin:15px}\n.pred{max-width:600px;margin:12px auto;text-align:left}.bg{height:20px;background:#444;border-radius:10px}.bar{height:100%;background:#00ff88;border-radius:10px}\n</style>\n</head>\n<body>\n<h1>🎵 Dance Pose Song Recognition</h1>\n<button id="start" onclick="init()">▶ Start Camera</button>\n<button id="stop" onclick="stopCamera()" disabled>⏹ Stop Camera</button>\n<p id="status">Camera not started</p>\n<canvas id="canvas"></canvas>\n<div id="song">Waiting...</div><div id="confidence">Confidence: 0%</div>\n<div id="predictions"></div>\n<script>\nconst URL="./my_model/";\nlet model,webcam,ctx,maxPredictions,running=false;\nasync function init(){\ntry{\ndocument.getElementById("status").innerText="Loading model...";\ndocument.getElementById("start").disabled=true;\nmodel=await tmPose.load(URL+"model.json",URL+"metadata.json");\nmaxPredictions=model.getTotalClasses();\nwebcam=new tmPose.Webcam(500,500,true);\nawait webcam.setup(); await webcam.play();\nconst canvas=document.getElementById("canvas");canvas.width=500;canvas.height=500;ctx=canvas.getContext("2d");\nconst box=document.getElementById("predictions");box.innerHTML="";\nfor(let i=0;i<maxPredictions;i++){\nlet d=document.createElement("div");d.className="pred";\nd.innerHTML=\'<div id="label-\'+i+\'"></div><div class="bg"><div class="bar" id="bar-\'+i+\'"></div></div>\';\nbox.appendChild(d);}\nrunning=true;document.getElementById("stop").disabled=false;\ndocument.getElementById("status").innerText="✅ Camera started - Show your pose";\nrequestAnimationFrame(loop);\n}catch(e){console.error(e);document.getElementById("status").innerText="❌ "+e.message;document.getElementById("start").disabled=false;}\n}\nasync function loop(){\nif(!running)return;webcam.update();await predict();requestAnimationFrame(loop);\n}\nasync function predict(){\nconst r=await model.estimatePose(webcam.canvas);\nconst prediction=await model.predict(r.posenetOutput);\nlet best=0,bestP=0;\nfor(let i=0;i<maxPredictions;i++){\nlet p=prediction[i].probability;\ndocument.getElementById("label-"+i).innerText=prediction[i].className+" : "+(p*100).toFixed(1)+"%";\ndocument.getElementById("bar-"+i).style.width=(p*100)+"%";\nif(p>bestP){bestP=p;best=i;}\n}\ndocument.getElementById("song").innerText=prediction[best].className;\ndocument.getElementById("confidence").innerText="Confidence: "+(bestP*100).toFixed(1)+"%";\nctx.drawImage(webcam.canvas,0,0);\nif(r.pose){tmPose.drawKeypoints(r.pose.keypoints,.5,ctx);tmPose.drawSkeleton(r.pose.keypoints,.5,ctx);}\n}\nfunction stopCamera(){\nrunning=false;if(webcam)webcam.stop();if(ctx)ctx.clearRect(0,0,500,500);\ndocument.getElementById("status").innerText="Camera stopped";\ndocument.getElementById("song").innerText="Waiting...";\ndocument.getElementById("confidence").innerText="Confidence: 0%";\ndocument.getElementById("start").disabled=false;document.getElementById("stop").disabled=true;\n}\n</script>\n</body>\n</html>\n'
html_path = BASE / 'index.html'
html_path.write_text(html, encoding='utf-8')
print('✅ Created:', html_path)


In [ ]:
import os, threading, http.server, socketserver, webbrowser

PORT=8000
os.chdir(BASE)
try:
    httpd
except NameError:
    httpd=socketserver.TCPServer(('127.0.0.1',PORT), http.server.SimpleHTTPRequestHandler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()

url=f'http://127.0.0.1:{PORT}/index.html'
print('✅ Server:', url)
webbrowser.open(url)


## How to use
1. Run the cells from top to bottom.
2. Chrome opens the local page.
3. Click **Start Camera** and allow camera access.
4. Show your full body to the camera.
5. The page displays the detected song/pose and confidence.

**Important:** Do not run old cells containing `import tensorflowjs as tfjs` or `tfjs.converters.load_keras_model(...)`.